# 第七章：进程并行

**系列：** OpenEvolve — 从零到专家·手撕全流程  
**上一章：** [06 — 基于 Diff 的进化](./06-diff-based-evolution.ipynb)  
**本章内容：** 为什么需要并行、进程 vs 线程、数据库快照、ProcessPoolExecutor、以及结果聚合。

---

## 1. 痛点：串行太慢了

到目前为止，我们的进化循环是这样的：

```
第 1 轮: 采样 → LLM生成(2s) → 评估(10s) → 存储   总计 ~12s
第 2 轮: 采样 → LLM生成(2s) → 评估(10s) → 存储   总计 ~12s
第 3 轮: 采样 → LLM生成(2s) → 评估(10s) → 存储   总计 ~12s
...
100 轮: 12s × 100 = 1200s = 20 分钟
```

但如果我们有 4 个 CPU 核心，为什么不**同时跑 4 个迭代**？

```
并行模式（4 workers）:
Worker 1: 第1轮(12s)  第5轮(12s)  第9轮(12s)  ...
Worker 2: 第2轮(12s)  第6轮(12s)  第10轮(12s) ...
Worker 3: 第3轮(12s)  第7轮(12s)  第11轮(12s) ...
Worker 4: 第4轮(12s)  第8轮(12s)  第12轮(12s) ...

100 轮: 12s × 25 = 300s = 5 分钟  (加速 4 倍！)
```

**生活类比：** 一个厨师做 100 道菜要 100 小时，4 个厨师各做 25 道只要 25 小时。

## 2. 理论：进程 vs 线程

### 2.1 Python 的 GIL 问题

Python 有一个**全局解释器锁（GIL）**，同一时刻只有一个线程能执行 Python 代码。  
这意味着多线程**无法**实现 CPU 密集型任务的真正并行。

| 方案 | CPU 密集型 | I/O 密集型 | 内存隔离 |
|------|-----------|-----------|----------|
| 多线程 (Thread) | 受 GIL 限制 | 有效 | 共享内存 |
| 多进程 (Process) | 真正并行 | 有效 | 独立内存 |
| asyncio | 无并行 | 非常高效 | 单进程 |

OpenEvolve 选择**多进程**，因为代码评估是 CPU 密集型的。

### 2.2 数据库快照模式

多进程的挑战：每个进程有**独立的内存空间**，不能直接共享数据库。

OpenEvolve 的解决方案——**快照模式**：

```mermaid
flowchart TB
    subgraph "主进程"
        DB[(数据库)] --> SNAP[创建快照]
        SNAP -->|序列化| W1
        SNAP -->|序列化| W2
        SNAP -->|序列化| W3
    end
    subgraph "Worker 进程"
        W1[Worker 1] -->|结果| AGG
        W2[Worker 2] -->|结果| AGG
        W3[Worker 3] -->|结果| AGG
    end
    AGG[结果聚合] --> DB
```

1. 主进程创建数据库的**不可变快照**
2. 快照通过序列化（pickle）发送给每个 Worker
3. Worker 在快照上独立工作
4. 结果序列化后返回主进程
5. 主进程将结果**合并**回数据库

### 2.3 加速公式

理想加速比：

$$S(n) = \frac{T_1}{T_n} = \frac{N \cdot t_{iter}}{\lceil N/n \rceil \cdot t_{iter} + n \cdot t_{overhead}} \approx n$$

其中 $N$ 是总迭代数，$n$ 是 Worker 数，$t_{iter}$ 是每次迭代时间，$t_{overhead}$ 是序列化开销。

**数值例子：**
> $N = 100$ 次迭代，$n = 4$ Workers，$t_{iter} = 12$ 秒，$t_{overhead} = 0.1$ 秒  
> $T_1 = 100 \times 12 = 1200$ 秒  
> $T_4 = 25 \times 12 + 4 \times 0.1 = 300.4$ 秒  
> $S(4) = 1200 / 300.4 \approx 3.99$  （接近 4 倍加速！）

**生活类比：** 快照就像考试时每人发一份试卷副本——大家独立作答，最后老师收卷统一批改。

## 3. 从零实现：可序列化结果

Worker 进程的结果需要通过**进程间通信**传回主进程。  
Python 的 `ProcessPoolExecutor` 使用 `pickle` 序列化，所以结果必须是可 pickle 的。

我们定义一个轻量级的结果数据类：

In [ ]:
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Any
import time


@dataclass
class WorkerResult:
    """Worker 进程的返回结果（可序列化）
    
    所有字段都是基本类型或 dict，确保 pickle 兼容。
    """
    # 子程序信息
    child_code: Optional[str] = None
    child_id: Optional[str] = None
    child_metrics: Optional[Dict[str, float]] = None
    parent_id: Optional[str] = None
    
    # 元数据
    iteration: int = 0
    iteration_time: float = 0.0
    target_island: Optional[int] = None
    
    # 错误处理
    error: Optional[str] = None
    
    @property
    def success(self) -> bool:
        return self.error is None and self.child_code is not None


# 验证可序列化
import pickle

result = WorkerResult(
    child_code="def sort_func(arr): return sorted(arr)",
    child_id="prog_001",
    child_metrics={"correctness": 1.0, "speed": 0.8},
    parent_id="prog_000",
    iteration=1,
    iteration_time=2.5,
    target_island=0,
)

# pickle 序列化/反序列化
data = pickle.dumps(result)
restored = pickle.loads(data)
print(f"序列化大小: {len(data)} bytes")
print(f"反序列化成功: {restored.success}")
print(f"子程序指标: {restored.child_metrics}")

## 4. 从零实现：数据库快照

快照是数据库在某一时刻的**不可变副本**，包含：
- 所有程序的代码和指标
- 岛屿分配信息
- 工件数据（限量，避免快照过大）

| 字段 | 内容 | 大小控制 |
|------|------|----------|
| programs | 所有程序 | 全部包含 |
| islands | 岛屿→程序ID映射 | 全部包含 |
| artifacts | 调试数据 | 限制数量（默认 100） |

In [ ]:
class SimpleDatabase:
    """简化版数据库，支持快照"""
    
    def __init__(self, num_islands: int = 3):
        self.programs = {}  # id -> {code, metrics, island, parent_id}
        self.islands = [[] for _ in range(num_islands)]  # 每个岛屿的程序ID列表
        self.best_score = 0.0
        self.best_id = None
    
    def add(self, program_id: str, code: str, metrics: dict,
            island: int = 0, parent_id: str = None):
        """添加程序到数据库"""
        self.programs[program_id] = {
            'code': code,
            'metrics': metrics,
            'island': island,
            'parent_id': parent_id,
        }
        self.islands[island % len(self.islands)].append(program_id)
        
        # 更新全局最优
        score = metrics.get('combined_score', 0)
        if score > self.best_score:
            self.best_score = score
            self.best_id = program_id
    
    def create_snapshot(self, max_artifacts: int = 100) -> dict:
        """创建数据库的不可变快照
        
        关键：返回的是深拷贝，Worker 的修改不会影响原数据库。
        """
        import copy
        return {
            'programs': copy.deepcopy(self.programs),
            'islands': [list(island) for island in self.islands],
            'best_score': self.best_score,
            'best_id': self.best_id,
        }
    
    def merge_result(self, result: WorkerResult):
        """将 Worker 结果合并回数据库"""
        if not result.success:
            return
        self.add(
            program_id=result.child_id,
            code=result.child_code,
            metrics=result.child_metrics,
            island=result.target_island or 0,
            parent_id=result.parent_id,
        )


# 演示快照
db = SimpleDatabase(num_islands=3)
db.add('p1', 'def f(): return 1', {'combined_score': 0.5}, island=0)
db.add('p2', 'def f(): return 2', {'combined_score': 0.7}, island=1)
db.add('p3', 'def f(): return 3', {'combined_score': 0.9}, island=2)

snapshot = db.create_snapshot()
print(f"快照包含 {len(snapshot['programs'])} 个程序")
print(f"快照岛屿: {[len(i) for i in snapshot['islands']]}")
print(f"快照最优: {snapshot['best_score']:.1f}")

# 修改快照不影响原数据库
snapshot['programs']['p1']['metrics']['combined_score'] = 999
print(f"\n修改快照后，原数据库 p1 分数: {db.programs['p1']['metrics']['combined_score']:.1f} (不受影响)")

## 5. 从零实现：Worker 函数

Worker 是运行在独立进程中的函数。它接收快照，完成一次完整的进化迭代：
1. 从快照中选择父程序
2. 变异生成子程序
3. 评估子程序
4. 返回 WorkerResult

**关键设计：** Worker 函数必须是**模块级函数**（不是类方法），因为 `pickle` 无法序列化局部函数。

In [ ]:
import random
import uuid


def evolution_worker(
    iteration: int,
    db_snapshot: dict,
    parent_id: str,
    target_island: int,
    mutate_fn=None,
    evaluate_fn=None,
) -> WorkerResult:
    """Worker 进程中运行的进化迭代
    
    注意：这是一个独立进程，不能访问主进程的任何对象。
    所有数据通过 db_snapshot 传入，结果通过 WorkerResult 返回。
    """
    start_time = time.time()
    
    try:
        # 1. 从快照获取父程序
        parent = db_snapshot['programs'].get(parent_id)
        if parent is None:
            return WorkerResult(error=f"Parent {parent_id} not found", iteration=iteration)
        
        parent_code = parent['code']
        
        # 2. 变异
        if mutate_fn:
            child_code = mutate_fn(parent_code)
        else:
            child_code = parent_code  # 无变异则复制
        
        # 3. 评估
        child_id = str(uuid.uuid4())[:8]
        if evaluate_fn:
            child_metrics = evaluate_fn(child_code)
        else:
            child_metrics = {'combined_score': 0.0}
        
        # 4. 返回结果
        return WorkerResult(
            child_code=child_code,
            child_id=child_id,
            child_metrics=child_metrics,
            parent_id=parent_id,
            iteration=iteration,
            iteration_time=time.time() - start_time,
            target_island=target_island,
        )
        
    except Exception as e:
        return WorkerResult(
            error=str(e),
            iteration=iteration,
            iteration_time=time.time() - start_time,
        )


# 测试 Worker（串行调用）
def simple_mutator(code):
    """简单变异：随机修改排序算法"""
    variants = [
        'def sort_func(arr): return sorted(arr)',
        'def sort_func(arr): return list(reversed(sorted(arr)))',  # bug: 降序
        'def sort_func(arr): return arr',  # bug: 不排序
        'def sort_func(arr):\n    arr = arr[:]\n    arr.sort()\n    return arr',
    ]
    return random.choice(variants)

def simple_evaluator(code):
    """简单评估"""
    try:
        ns = {}
        exec(code, ns)
        f = ns.get('sort_func')
        if not f:
            return {'combined_score': 0.0}
        tests = [([3,1,2], [1,2,3]), ([5,4,3,2,1], [1,2,3,4,5]), ([], [])]
        passed = sum(1 for i, e in tests if f(i[:]) == e)
        return {'combined_score': passed / len(tests)}
    except:
        return {'combined_score': 0.0}


# 单个 Worker 测试
snapshot = db.create_snapshot()
# 为 snapshot 添加一个排序程序
snapshot['programs']['sort_parent'] = {
    'code': 'def sort_func(arr): return sorted(arr)',
    'metrics': {'combined_score': 1.0},
    'island': 0,
    'parent_id': None,
}

result = evolution_worker(
    iteration=0,
    db_snapshot=snapshot,
    parent_id='sort_parent',
    target_island=0,
    mutate_fn=simple_mutator,
    evaluate_fn=simple_evaluator,
)
print(f"Worker 结果: success={result.success}, score={result.child_metrics}")
print(f"耗时: {result.iteration_time:.4f}s")

## 6. 从零实现：并行控制器

并行控制器负责：
1. 管理 Worker 进程池
2. 分发迭代任务到各岛屿
3. 收集结果并合并到数据库
4. 处理超时和错误

### 设计决策

| 问题 | OpenEvolve 的选择 | 原因 |
|------|-------------------|------|
| 进程池类型 | ProcessPoolExecutor | 简单易用、标准库 |
| 任务分配 | 轮询各岛屿 | 保持岛屿间均匀探索 |
| 快照时机 | 每批次创建新快照 | 确保 Worker 看到最新状态 |
| 错误处理 | 记录并继续 | 单个失败不应停止进化 |

In [ ]:
from concurrent.futures import ProcessPoolExecutor, as_completed, Future
import os


class ParallelEvolutionController:
    """并行进化控制器
    
    使用 ProcessPoolExecutor 管理多个 Worker 进程，
    将进化迭代分布到不同岛屿上并行执行。
    """
    
    def __init__(
        self,
        database: SimpleDatabase,
        num_workers: int = None,
        mutate_fn=None,
        evaluate_fn=None,
    ):
        self.database = database
        self.num_workers = num_workers or min(4, os.cpu_count() or 1)
        self.mutate_fn = mutate_fn
        self.evaluate_fn = evaluate_fn
        self.stats = {
            'total_iterations': 0,
            'successful': 0,
            'failed': 0,
            'total_time': 0.0,
        }
    
    def run_batch(self, batch_size: int = None) -> List[WorkerResult]:
        """运行一批并行迭代
        
        注意：这里用线程池模拟进程池行为（因为 Jupyter 中 ProcessPoolExecutor
        需要 if __name__ == '__main__' 保护，且函数必须在模块级别定义）。
        实际 OpenEvolve 使用 ProcessPoolExecutor。
        """
        from concurrent.futures import ThreadPoolExecutor
        
        if batch_size is None:
            batch_size = self.num_workers
        
        # 1. 创建快照
        snapshot = self.database.create_snapshot()
        
        # 2. 为每个任务选择父程序和目标岛屿（轮询分配）
        tasks = []
        num_islands = len(self.database.islands)
        
        for i in range(batch_size):
            island_id = i % num_islands
            island_programs = snapshot['islands'][island_id]
            
            if island_programs:
                parent_id = random.choice(island_programs)
            elif snapshot['programs']:
                parent_id = random.choice(list(snapshot['programs'].keys()))
            else:
                continue
            
            tasks.append((i, snapshot, parent_id, island_id))
        
        # 3. 并行执行
        results = []
        batch_start = time.time()
        
        with ThreadPoolExecutor(max_workers=self.num_workers) as executor:
            futures = {}
            for iteration, snap, parent_id, island_id in tasks:
                future = executor.submit(
                    evolution_worker,
                    iteration=self.stats['total_iterations'] + iteration,
                    db_snapshot=snap,
                    parent_id=parent_id,
                    target_island=island_id,
                    mutate_fn=self.mutate_fn,
                    evaluate_fn=self.evaluate_fn,
                )
                futures[future] = iteration
            
            # 4. 收集结果（as_completed = 谁先完成先处理谁）
            for future in as_completed(futures):
                try:
                    result = future.result(timeout=30)
                    results.append(result)
                    
                    if result.success:
                        self.stats['successful'] += 1
                    else:
                        self.stats['failed'] += 1
                except Exception as e:
                    self.stats['failed'] += 1
                    results.append(WorkerResult(error=str(e)))
        
        batch_time = time.time() - batch_start
        self.stats['total_iterations'] += batch_size
        self.stats['total_time'] += batch_time
        
        # 5. 合并结果到数据库
        for result in results:
            self.database.merge_result(result)
        
        return results


print(f"ParallelEvolutionController 定义完成 ✓")
print(f"可用 CPU 核心数: {os.cpu_count()}")

## 7. 运行并行进化演示

用并行控制器跑 10 批次（每批 4 个 Worker），观察进化过程。

In [ ]:
# 准备数据库
parallel_db = SimpleDatabase(num_islands=3)

# 种子程序
seeds = [
    ('seed_0', 'def sort_func(arr): return sorted(arr)', {'combined_score': 1.0}, 0),
    ('seed_1', 'def sort_func(arr): return arr', {'combined_score': 0.0}, 1),
    ('seed_2', 'def sort_func(arr): return list(reversed(arr))', {'combined_score': 0.0}, 2),
]
for pid, code, metrics, island in seeds:
    parallel_db.add(pid, code, metrics, island)

# 创建控制器
controller = ParallelEvolutionController(
    database=parallel_db,
    num_workers=4,
    mutate_fn=simple_mutator,
    evaluate_fn=simple_evaluator,
)

# 运行 10 批次
random.seed(42)
batch_scores = []

for batch in range(10):
    results = controller.run_batch(batch_size=4)
    
    successes = [r for r in results if r.success]
    if successes:
        batch_best = max(r.child_metrics['combined_score'] for r in successes)
    else:
        batch_best = 0.0
    batch_scores.append(parallel_db.best_score)
    
    if (batch + 1) % 5 == 0:
        print(f"批次 {batch+1:>3}: 成功={len(successes)}/4, "
              f"批次最优={batch_best:.2f}, 全局最优={parallel_db.best_score:.2f}, "
              f"总程序数={len(parallel_db.programs)}")

print(f"\n=== 统计 ===")
print(f"总迭代: {controller.stats['total_iterations']}")
print(f"成功: {controller.stats['successful']}")
print(f"失败: {controller.stats['failed']}")
print(f"总耗时: {controller.stats['total_time']:.2f}s")
print(f"平均每迭代: {controller.stats['total_time']/controller.stats['total_iterations']*1000:.1f}ms")

## 8. 对比：串行 vs 并行

让我们量化并行带来的加速。

In [ ]:
import time as time_module

def timed_serial_evolution(db, mutate_fn, evaluate_fn, iterations=40):
    """串行进化"""
    start = time_module.perf_counter()
    for i in range(iterations):
        # 随机选一个程序
        parent_id = random.choice(list(db.programs.keys()))
        parent = db.programs[parent_id]
        
        child_code = mutate_fn(parent['code'])
        child_metrics = evaluate_fn(child_code)
        child_id = f"serial_{i}"
        db.add(child_id, child_code, child_metrics, 
               island=parent.get('island', 0), parent_id=parent_id)
    
    return time_module.perf_counter() - start


def timed_parallel_evolution(db, mutate_fn, evaluate_fn, iterations=40, workers=4):
    """并行进化"""
    ctrl = ParallelEvolutionController(
        database=db, num_workers=workers,
        mutate_fn=mutate_fn, evaluate_fn=evaluate_fn,
    )
    start = time_module.perf_counter()
    num_batches = iterations // workers
    for _ in range(num_batches):
        ctrl.run_batch(batch_size=workers)
    return time_module.perf_counter() - start


# 准备两个相同的数据库
random.seed(42)
db_serial = SimpleDatabase(num_islands=3)
db_serial.add('s0', 'def sort_func(arr): return sorted(arr)', {'combined_score': 1.0}, 0)

db_parallel = SimpleDatabase(num_islands=3)
db_parallel.add('s0', 'def sort_func(arr): return sorted(arr)', {'combined_score': 1.0}, 0)

# 添加模拟的计算延迟
def slow_evaluator(code):
    time_module.sleep(0.05)  # 模拟 50ms 评估时间
    return simple_evaluator(code)

# 对比
iters = 40
serial_time = timed_serial_evolution(db_serial, simple_mutator, slow_evaluator, iters)
parallel_time = timed_parallel_evolution(db_parallel, simple_mutator, slow_evaluator, iters, workers=4)

speedup = serial_time / parallel_time if parallel_time > 0 else 0

print(f"=== {iters} 次迭代对比 ===")
print(f"串行耗时:   {serial_time:.2f}s")
print(f"并行耗时:   {parallel_time:.2f}s  ({4} workers)")
print(f"加速比:     {speedup:.1f}x")
print(f"理论加速比: {4.0}x")
print(f"并行效率:   {speedup/4*100:.0f}%")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = ['SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 图1：加速比对比
ax1 = axes[0]
workers_list = [1, 2, 4]
# 理论加速（线性）
ax1.plot(workers_list, workers_list, 'k--', label='Ideal', linewidth=2)
# 实际加速（模拟）
actual_speedups = [1.0, 1.8, speedup]  # 近似值
ax1.plot(workers_list, actual_speedups, 'bo-', label='Actual', linewidth=2, markersize=8)
ax1.set_xlabel('Number of Workers', fontsize=12)
ax1.set_ylabel('Speedup', fontsize=12)
ax1.set_title('Parallelism Speedup', fontsize=14)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# 图2：全局最优分数随批次变化
ax2 = axes[1]
ax2.plot(range(1, len(batch_scores)+1), batch_scores, 'g-o', linewidth=2, markersize=5)
ax2.set_xlabel('Batch', fontsize=12)
ax2.set_ylabel('Best Score', fontsize=12)
ax2.set_title('Best Score over Batches', fontsize=14)
ax2.grid(True, alpha=0.3)
ax2.set_ylim(-0.05, 1.05)

# 图3：岛屿程序分布
ax3 = axes[2]
island_counts = [len(island) for island in parallel_db.islands]
colors = ['#e74c3c', '#3498db', '#2ecc71']
bars = ax3.bar(range(len(island_counts)), island_counts, color=colors)
ax3.set_xlabel('Island', fontsize=12)
ax3.set_ylabel('Programs', fontsize=12)
ax3.set_title('Programs per Island', fontsize=14)
for bar, count in zip(bars, island_counts):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             str(count), ha='center', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

## 9. 错误处理与容错

在并行环境中，错误处理尤为重要：

| 错误类型 | 处理方式 | 原因 |
|---------|---------|------|
| Worker 异常 | 记录错误，继续运行 | 一个 Worker 失败不应影响其他 |
| 超时 | 取消 Future，记录日志 | 防止 Worker 挂起 |
| 序列化失败 | 跳过该结果 | 结果太大或包含不可序列化对象 |

OpenEvolve 中的关键容错设计：

```python
# 超时保护 = 评估超时 + 30 秒缓冲
timeout_seconds = config.evaluator.timeout + 30
result = future.result(timeout=timeout_seconds)
```

**生活类比：** 就像外卖平台——如果某个骑手出了问题（车坏了、迷路了），  
平台不会暂停所有订单，而是记录问题、重新分配，其他骑手继续送餐。

## 10. 我们的实现 vs. OpenEvolve 源码

| 概念 | 我们的实现 | OpenEvolve 源码 | 差异说明 |
|------|-----------|----------------|----------|
| Worker 结果 | `WorkerResult` | `SerializableResult` | 字段稍有不同 |
| 数据库快照 | `SimpleDatabase.create_snapshot()` | `process_parallel.py:L442-470` | 源码含工件限制 |
| Worker 函数 | `evolution_worker()` | `_run_iteration_worker()` | 源码含 LLM 和评估器初始化 |
| 进程池 | `ThreadPoolExecutor`（演示） | `ProcessPoolExecutor` | 源码用真正多进程 |
| 任务分配 | 轮询岛屿 | 轮询岛屿 | 逻辑一致 |
| 结果聚合 | `merge_result()` | `database.add()` | 源码含 MAP-Elites 放置 |
| 惰性初始化 | 未实现 | `_lazy_init_worker_components()` | 源码延迟创建 LLM/评估器 |
| Worker 初始化 | 未实现 | `_worker_init()` | 源码在进程启动时初始化环境 |

### 关键配置项

```yaml
evaluator:
  parallel_evaluations: 4    # Worker 进程数
  timeout: 300               # 评估超时（秒）

database:
  num_islands: 5              # 岛屿数
  max_snapshot_artifacts: 100 # 快照中的最大工件数

# Python 3.11+ 可选
max_tasks_per_child: null    # Worker 进程最大任务数（防内存泄漏）
```

### 关键源码位置

| 功能 | 文件 | 行号 |
|------|------|------|
| ProcessPoolExecutor 配置 | `process_parallel.py` | L396-425 |
| 数据库快照 | `process_parallel.py` | L442-470 |
| Worker 函数 | `process_parallel.py` | L134-332 |
| Worker 初始化 | `process_parallel.py` | L39-95 |
| 惰性初始化 | `process_parallel.py` | L98-131 |
| TaskPool（异步并发限制） | `async_utils.py` | L167-228 |
| 岛屿安全采样 | `database.py` | L403-470 |
| 迭代提交 | `process_parallel.py` | L796-827 |
| 结果聚合 | `process_parallel.py` | L531-665 |

## 11. 本章总结

```mermaid
flowchart TB
    subgraph "主进程"
        DB[(数据库)] --> SNAP[创建快照]
        SNAP --> DIST[任务分配<br/>轮询各岛屿]
        AGG[结果聚合] --> DB
    end
    
    subgraph "Worker 进程池"
        DIST --> W1[Worker 1<br/>Island 0]
        DIST --> W2[Worker 2<br/>Island 1]
        DIST --> W3[Worker 3<br/>Island 2]
        DIST --> W4[Worker 4<br/>Island 0]
        W1 -->|WorkerResult| AGG
        W2 -->|WorkerResult| AGG
        W3 -->|WorkerResult| AGG
        W4 -->|WorkerResult| AGG
    end
```

### 核心收获

| 概念 | 一句话总结 |
|------|----------|
| 进程并行 | 绕过 GIL，用多进程实现 CPU 密集型任务的真正并行 |
| 数据库快照 | 不可变副本发给 Worker，避免共享内存冲突 |
| 轮询分配 | 任务均匀分布到各岛屿，保持多样性 |
| 结果聚合 | Worker 返回可序列化结果，主进程统一合并 |
| 容错设计 | 单个 Worker 失败不影响整体进化 |

### 下一章预告

并行进化可以跑得很快，但如果进化跑了 1000 轮然后程序崩溃了怎么办？  
所有进度都丢失了！下一章我们实现 **检查点机制** —— 定期保存进化状态，支持断点续跑。

## 12. 保存模块

将本章实现的并行控制器保存到 `our-implementation/` 目录。

In [ ]:
save_code = '''
"""并行进化控制器 - 第七章实现

基于 ProcessPoolExecutor 的进程级并行进化。
"""
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Any
from concurrent.futures import ProcessPoolExecutor, as_completed
import time
import os
import uuid
import random


@dataclass
class WorkerResult:
    child_code: Optional[str] = None
    child_id: Optional[str] = None
    child_metrics: Optional[Dict[str, float]] = None
    parent_id: Optional[str] = None
    iteration: int = 0
    iteration_time: float = 0.0
    target_island: Optional[int] = None
    error: Optional[str] = None
    
    @property
    def success(self):
        return self.error is None and self.child_code is not None


def evolution_worker(iteration, db_snapshot, parent_id, target_island,
                     mutate_fn=None, evaluate_fn=None):
    start_time = time.time()
    try:
        parent = db_snapshot["programs"].get(parent_id)
        if parent is None:
            return WorkerResult(error=f"Parent {parent_id} not found")
        child_code = mutate_fn(parent["code"]) if mutate_fn else parent["code"]
        child_id = str(uuid.uuid4())[:8]
        child_metrics = evaluate_fn(child_code) if evaluate_fn else {"combined_score": 0.0}
        return WorkerResult(
            child_code=child_code, child_id=child_id,
            child_metrics=child_metrics, parent_id=parent_id,
            iteration=iteration, iteration_time=time.time()-start_time,
            target_island=target_island,
        )
    except Exception as e:
        return WorkerResult(error=str(e), iteration=iteration)
'''

import os
os.makedirs('../our-implementation', exist_ok=True)
with open('../our-implementation/parallel.py', 'w', encoding='utf-8') as f:
    f.write(save_code)

print('已保存到 our-implementation/parallel.py ✓')